# Übung 04: SQL & PostgreSQL

**Übungsnotebook zur Vorbereitung auf LN 1 (Mo. 09.11.2026)** · Cheatsheet: **Kapitel 4 (SQL & PostgreSQL)** und Kapitel 11 (Checklisten)

| | |
|---|---|
| **Themen** | JOIN-Typen, NULL-Logik, Aggregation & HAVING, Fan-out, Anti-Join/NOT EXISTS, CASE, CTE, Window-Funktionen, DDL & Constraints, Datenqualität |
| **Format** | wie die Musterprüfungen AP01–AP03: Teil A Konzeptfragen · Teil B Fehleranalyse · Teil C Code schreiben |
| **Zeitbudget** | ca. 250 Minuten (Prüfungstempo: ca. 4.5 Min. pro Punkt) |
| **Lösungen** | keine – alles ist mit dem Cheatsheet bzw. den Kursunterlagen lösbar. Hinweise wie *(→ 4.5)* verweisen auf den Cheatsheet-Abschnitt. |

**Szenario:** Ein fiktiver Bergsportverband betreibt ein Reservationssystem für seine Berghütten (Saison Juni–Oktober 2026). Das System ist historisch gewachsen: nicht alle Regeln sind als Constraints hinterlegt.

**So üben Sie prüfungsnah**
- Copilot/KI-Autocomplete ausschalten, wie in der Prüfung (Cheatsheet 1.1).
- Begründungen immer **mit konkretem Bezug zu den Daten** (Spaltennamen, Werte, Zahlen aus dem Output) – sonst gibt es in der Prüfung 0 Punkte.
- Verändern Sie die Ausgangsdaten nicht, sondern arbeiten Sie pro Aufgabe mit eigenen Variablen (z.B. `df_B1`, `df_C2`).
- Am Schluss: *Kernel → Restart & Run All* (Cheatsheet 11.5).

**Datenbank:** Das Notebook verwendet automatisch die PostgreSQL-Datenbank Ihres Codespaces (wie AP01). Ist sie nicht erreichbar, wird ein SQLite-Backup erzeugt. Alle verlangten Abfragen lassen sich so schreiben, dass sie in **beiden** Datenbanken laufen – PostgreSQL-spezifische Funktionen (`ILIKE`, `::numeric`, `DATE_TRUNC`, `INITCAP`) funktionieren nur mit PostgreSQL.

## Setup (wird nicht bewertet)

Die folgende Zelle erzeugt die Übungsdaten **synthetisch und reproduzierbar** (fixer Seed) im Unterordner neben diesem Notebook. Sie brauchen den Code nicht zu verstehen und sollen ihn **nicht verändern** – behandeln Sie die erzeugten Dateien so, als hätten Sie sie in der Prüfung erhalten. *(Natürlich dürfen Sie die Dateien im Editor öffnen und ansehen.)*

In [ ]:
# Libraries
import os
import pandas as pd
from sqlalchemy import create_engine, text

# Settings
import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

# Get current working directory
print(os.getcwd())

In [ ]:
# ---------------------------------------------------------------
# Database connection: PostgreSQL if available, otherwise SQLite
# ---------------------------------------------------------------
user     = 'pgadmin'
password = 'geheim'
port     = '5432'
database = 'postgres'

db_connection_url = None
for candidate in ('db', 'localhost'):
    try:
        probe = create_engine(f'postgresql://{user}:{password}@{candidate}:{port}/{database}',
                              connect_args={'connect_timeout': 3})
        with probe.connect() as connection:
            connection.execute(text('SELECT 1'))
        probe.dispose()
        db_connection_url = f'postgresql://{user}:{password}@{candidate}:{port}/{database}'
        break
    except Exception:
        pass

if db_connection_url is None:
    os.makedirs('data_uebung04', exist_ok=True)
    db_connection_url = 'sqlite:///data_uebung04/huetten.db'
    print('PostgreSQL nicht erreichbar -> SQLite-Backup wird verwendet')

print('Connection URL:', db_connection_url.replace(password, '***'))

In [ ]:
# ---------------------------------------------------------------
# Data generator (do not modify): creates and fills the tables
# huts, guests, bookings, payments
# ---------------------------------------------------------------
import numpy as np
from datetime import date, timedelta

rng = np.random.default_rng(404)

huts = pd.DataFrame({
    'hutid': [1, 2, 3, 4, 5, 6, 7, 8],
    'name': ['Tierbergli', 'Cabane de Moiry', 'Kesch-Hütte', 'Lämmerenhütte',
             'Capanna Cristallina', 'Glärnischhütte', 'Bergseehütte', 'Chamanna Nova'],
    'canton': ['BE', 'VS', 'GR', 'VS', 'TI', 'GL', 'UR', 'GR'],
    'altitude_m': [2795, 2825, 2632, 2501, 2575, 1990, 2370, 2410],
    'beds': [70, 100, 92, 96, 120, 50, 60, 40]})

first = ['Anna', 'Luca', 'Mia', 'Noah', 'Lea', 'Elias', 'Lina', 'Leon', 'Sara', 'David', 'Nina', 'Jan',
         'Laura', 'Tim', 'Julia', 'Marco', 'Sophie', 'Reto', 'Claudia', 'Urs']
last = ['Müller', 'Meier', 'Schmid', 'Keller', 'Weber', 'Huber', 'Schneider', 'Steiner', 'Brunner',
        'Baumann', 'Gerber', 'Frei', 'Moser', 'Zürcher', 'Graf', 'Wyss']
n_guests = 420
mt_clean = rng.choice(['Mitglied', 'Gast', 'Jugend'], size=n_guests, p=[0.55, 0.35, 0.10]).astype(object)
dirty_idx = rng.choice(n_guests, size=19, replace=False)
dirty_vals = ['mitglied', 'MITGLIED', 'Mitgleid', 'Mitglied ', 'gast', 'Gast ', 'jugend', None, None, None,
              'Mitglied', 'Gast', 'mitglied', 'Mitgleid', 'GAST', None, 'Senior', 'Senior', 'mitglied']
mt_clean[dirty_idx] = dirty_vals
fn = rng.choice(first, n_guests)
ln = rng.choice(last, n_guests)
emails = [f'{f.lower()}.{l.lower().replace("ü", "ue").replace("ä", "ae")}{i}@example.ch' for i, (f, l) in enumerate(zip(fn, ln))]
emails = np.array(emails, dtype=object)
emails[rng.choice(n_guests, size=31, replace=False)] = None
emails[17] = emails[18] = 'familie.keller@example.ch'
guests = pd.DataFrame({'guestid': np.arange(1, n_guests + 1), 'firstname': fn, 'lastname': ln,
                       'member_type': mt_clean, 'email': emails,
                       'birthyear': rng.integers(1945, 2010, n_guests)})

# bookings: 330 of 420 guests book something; hut 8 (new) has no bookings at all
bookers = rng.choice(np.arange(1, n_guests + 1), size=330, replace=False)
popularity = np.array([0.15, 0.17, 0.14, 0.16, 0.13, 0.12, 0.13])
rows = []
season_start = date(2026, 6, 15)
for bid in range(1, 1601):
    gid = int(rng.choice(bookers))
    hid = int(rng.choice(np.arange(1, 8), p=popularity / popularity.sum()))
    arr = season_start + timedelta(days=int(rng.integers(0, 110)))
    status = rng.choice(['confirmed', 'cancelled', 'no-show'], p=[0.80, 0.15, 0.05])
    rows.append((bid, gid, hid, arr.isoformat(), int(rng.choice([1, 1, 1, 2, 2, 3])),
                 int(rng.choice([1, 2, 2, 2, 3, 4, 4, 6, 8])), str(status)))
bookings = pd.DataFrame(rows, columns=['bookingid', 'guestid', 'hutid', 'arrival_date', 'nights', 'persons', 'status'])
bookings['guestid'] = bookings['guestid'].astype('Int64')
walkin = rng.choice(bookings.index, size=12, replace=False)
bookings.loc[walkin, 'guestid'] = pd.NA                  # walk-in guests without guest record
bookings.loc[bookings.index[-1], 'guestid'] = 9999        # orphan: guest record was deleted

# payments: 1 payment, or deposit + rest (2 payments) for ~35 % of the paid bookings
price = {1: 62, 2: 58, 3: 60, 4: 65, 5: 55, 6: 52, 7: 57}
pay_rows, pid = [], 1
for b in bookings.itertuples():
    if b.status == 'cancelled':
        continue
    total = price[b.hutid] * b.nights * b.persons
    pay_date = date.fromisoformat(b.arrival_date)
    if rng.random() < 0.35:
        dep = round(total * 0.3, 2)
        pay_rows.append((pid, b.bookingid, dep, (pay_date - timedelta(days=30)).isoformat(), 'TWINT')); pid += 1
        pay_rows.append((pid, b.bookingid, round(total - dep, 2), pay_date.isoformat(), 'Karte')); pid += 1
    else:
        pay_rows.append((pid, b.bookingid, float(total), pay_date.isoformat(), str(rng.choice(['Karte', 'Bar', 'TWINT'])))); pid += 1
payments = pd.DataFrame(pay_rows, columns=['paymentid', 'bookingid', 'amount_chf', 'paid_at', 'method'])

# --- create schema (identical in PostgreSQL and SQLite) and load the data
ddl = [
    'DROP TABLE IF EXISTS reviews',
    'DROP TABLE IF EXISTS payments',
    'DROP TABLE IF EXISTS bookings',
    'DROP TABLE IF EXISTS guests',
    'DROP TABLE IF EXISTS huts',
    """CREATE TABLE huts (hutid INTEGER PRIMARY KEY, name VARCHAR(50) NOT NULL, canton CHAR(2),
                          altitude_m INTEGER, beds INTEGER)""",
    """CREATE TABLE guests (guestid INTEGER PRIMARY KEY, firstname VARCHAR(50), lastname VARCHAR(50),
                            member_type VARCHAR(20), email VARCHAR(100), birthyear INTEGER)""",
    """CREATE TABLE bookings (bookingid INTEGER PRIMARY KEY, guestid INTEGER,
                              hutid INTEGER REFERENCES huts(hutid), arrival_date VARCHAR(10),
                              nights INTEGER, persons INTEGER, status VARCHAR(20))""",
    """CREATE TABLE payments (paymentid INTEGER PRIMARY KEY, bookingid INTEGER REFERENCES bookings(bookingid),
                              amount_chf DOUBLE PRECISION, paid_at VARCHAR(10), method VARCHAR(10))""",
]
engine = create_engine(db_connection_url)
with engine.begin() as connection:
    for stmt in ddl:
        connection.execute(text(stmt))
for name, df_t in [('huts', huts), ('guests', guests), ('bookings', bookings), ('payments', payments)]:
    df_t.to_sql(name, engine, if_exists='append', index=False)
    print(f'{name:<10}{len(df_t):>6} Zeilen')
engine.dispose()

stays = bookings.loc[bookings['status'] == 'confirmed']
print('\nKontrollwert Hüttenstatistik: Total Übernachtungen (persons × nights, nur confirmed):',
      int((stays['persons'] * stays['nights']).sum()))

### Tabellen

| Tabelle | Spalten | Hinweise |
|---|---|---|
| `huts` | `hutid` (PK), `name`, `canton`, `altitude_m`, `beds` | 8 Hütten; die *Chamanna Nova* ist neu |
| `guests` | `guestid` (PK), `firstname`, `lastname`, `member_type`, `email`, `birthyear` | gültige `member_type`: `Mitglied`, `Gast`, `Jugend` |
| `bookings` | `bookingid` (PK), `guestid`, `hutid` (FK), `arrival_date` (Text `JJJJ-MM-TT`), `nights`, `persons`, `status` | `status`: `confirmed`, `cancelled`, `no-show`; `guestid` ist bei Spontangästen (walk-in) leer |
| `payments` | `paymentid` (PK), `bookingid` (FK), `amount_chf`, `paid_at`, `method` | eine Buchung kann **mehrere** Zahlungen haben (Anzahlung + Rest) |

Die erste Zelle unten zeigt Ihnen die Spalten jeder Tabelle; verschaffen Sie sich zuerst einen Überblick (`SELECT * ... LIMIT 5`).

In [ ]:
# Overview: first rows of each table
engine = create_engine(db_connection_url)
for t in ['huts', 'guests', 'bookings', 'payments']:
    print(f'--- {t}')
    display(pd.read_sql_query(f'SELECT * FROM {t} LIMIT 3', con=engine))
engine.dispose()

---

<div style="background-color:#2E4057; color:white; padding:16px 24px; border-radius:6px; font-size:18px; margin-top:8px;">
&#9654;&nbsp; <strong>Ab hier beginnen die Übungsaufgaben — 56 Punkte</strong><br>
<span style="font-size:15px; font-weight:normal;">Tragen Sie Ihre Antworten direkt in die vorgesehenen Zellen ein und führen Sie alle Code-Zellen aus.</span>
</div>

---
## Teil A – Konzeptfragen (12 Punkte)

*Antworten ohne Bezug zum Schema der Hüttendatenbank werden nicht gewertet.*

### A1 (4 Punkte)

Ein Entwickler schlägt vor, die Tabelle `payments` abzuschaffen und stattdessen eine Spalte `total_paid` in `bookings` zu führen: *«Das spart einen JOIN.»*

Beurteilen Sie den Vorschlag: Welche Kardinalität besteht zwischen `bookings` und `payments`? Welche Information ginge verloren, und welches Risiko (Stichwort Anomalie/Redundanz) entsteht, wenn man **zusätzlich** zu `payments` eine Spalte `total_paid` führen würde? Begründen Sie mit einem konkreten Szenario aus dem Hüttenbetrieb. *(→ 4.9)*

**Ihre Antwort:**

> [Hier eintragen]

### A2 (4 Punkte)

Die folgende Abfrage liefert **eine** Zeile:

```sql
SELECT b.bookingid, b.guestid
FROM bookings b
LEFT JOIN guests g ON b.guestid = g.guestid
WHERE b.guestid IS NOT NULL AND g.guestid IS NULL;
```

a) Was bedeutet dieses Ergebnis fachlich, und welche Data-Quality-Dimension ist verletzt?

b) Welcher Constraint hätte das verhindert, und warum fehlt er im aktuellen Schema vermutlich? *(Tipp: Sehen Sie sich die `CREATE TABLE`-Anweisungen im Setup an.)*

c) Welche Auswirkung hat diese Zeile auf eine Umsatzauswertung, die `bookings` mit `guests` per **INNER JOIN** verbindet? *(→ 4.5, 4.8, 4.9)*

**Ihre Antwort:**

> [Hier eintragen]

### A3 (4 Punkte)

Führen Sie die folgende Abfrage aus und erklären Sie, **was jede der drei Zahlen misst** und was die Unterschiede zwischen ihnen über die Datenqualität der Tabelle `guests` aussagen (mit den konkreten Zahlen). *(→ 4.4)*

```sql
SELECT COUNT(*) AS n_rows, COUNT(email) AS n_email, COUNT(DISTINCT email) AS n_distinct_email
FROM guests;
```

In [ ]:
engine = create_engine(db_connection_url)

df_A3 = pd.read_sql_query("""
-- SQL-Abfrage aus der Aufgabenstellung hier eintragen

""", con=engine)

engine.dispose()
df_A3

**Ihre Antwort:**

> [Hier eintragen]

---
## Teil B – SQL-Fehleranalyse (24 Punkte)

*Jede Abfrage läuft fehlerfrei durch, liefert aber ein falsches oder irreführendes Geschäftsergebnis. Führen Sie die Originalabfrage aus, um das tatsächliche Ergebnis zu sehen.*

### B1 (8 Punkte)

**Ziel:** Anzahl **bestätigter** Buchungen (`status = 'confirmed'`) pro Hütte – für **alle 8 Hütten**, Hütten ohne bestätigte Buchung mit `0`, absteigend sortiert.

In [ ]:
# Original query (do not modify this cell)
engine = create_engine(db_connection_url)
df_B1_orig = pd.read_sql_query("""
SELECT h.name, COUNT(*) AS n_confirmed
FROM huts h
LEFT JOIN bookings b ON h.hutid = b.hutid
WHERE b.status = 'confirmed'
GROUP BY h.name
ORDER BY n_confirmed DESC;
""", con=engine)
engine.dispose()
print(len(df_B1_orig), 'Zeilen')
df_B1_orig

**Aufgabe:** Erklären Sie, warum die Abfrage trotz `LEFT JOIN` nicht alle Hütten liefert. Schreiben Sie die korrigierte Abfrage und begründen Sie **jede** Änderung. Überlegen Sie auch: Würde Ihre Korrektur noch stimmen, wenn Sie nur die `WHERE`-Bedingung verschieben, aber `COUNT(*)` beibehalten? *(→ 4.5, 11.1)*

**Begründung der Änderungen:**

> [Hier eintragen]

**Korrigierte SQL-Abfrage:**

In [ ]:
engine = create_engine(db_connection_url)

df_B1 = pd.read_sql_query("""
-- Korrigierte SQL-Abfrage hier eintragen

""", con=engine)

engine.dispose()
df_B1

### B2 (8 Punkte)

**Ziel:** Pro Hütte die Anzahl **Übernachtungen** (`persons × nights`) und den **bezahlten Umsatz** (Summe `amount_chf`) – nur bestätigte Buchungen. Die Summe der Übernachtungen über alle Hütten muss dem **Kontrollwert der Hüttenstatistik** entsprechen (Ausgabe im Setup).

In [ ]:
# Original query (do not modify this cell)
engine = create_engine(db_connection_url)
df_B2_orig = pd.read_sql_query("""
SELECT h.name,
       SUM(b.persons * b.nights) AS overnight_stays,
       SUM(p.amount_chf)         AS revenue_chf
FROM bookings b
JOIN huts h     ON b.hutid = h.hutid
JOIN payments p ON p.bookingid = b.bookingid
WHERE b.status = 'confirmed'
GROUP BY h.name
ORDER BY overnight_stays DESC;
""", con=engine)
engine.dispose()
print('Total Übernachtungen:', df_B2_orig['overnight_stays'].sum())
df_B2_orig

**Aufgabe:** Erklären Sie, warum die Übernachtungen zu hoch sind, der Umsatz aber stimmt. Schreiben Sie eine korrigierte Abfrage, die **beide** Kennzahlen korrekt liefert (Tipp: zuerst aggregieren, dann verbinden). Prüfen Sie Ihr Ergebnis gegen den Kontrollwert und begründen Sie jede Änderung. *(→ 4.5 «Fan-out», 4.6)*

**Begründung der Änderungen:**

> [Hier eintragen]

**Korrigierte SQL-Abfrage:**

In [ ]:
engine = create_engine(db_connection_url)

df_B2 = pd.read_sql_query("""
-- Korrigierte SQL-Abfrage hier eintragen

""", con=engine)

engine.dispose()
df_B2

### B3 (8 Punkte)

**Ziel:** Für eine Reaktivierungs-Kampagne: alle Gäste (`guestid`, `firstname`, `lastname`, `email`), die **keine bestätigte** Buchung haben – also Gäste ganz ohne Buchung **und** Gäste, deren Buchungen alle storniert oder `no-show` waren. Laut CRM betrifft das gut 90 Gäste.

In [ ]:
# Original query (do not modify this cell)
engine = create_engine(db_connection_url)
df_B3_orig = pd.read_sql_query("""
SELECT g.guestid, g.firstname, g.lastname, g.email
FROM guests g
WHERE g.guestid NOT IN (SELECT b.guestid
                        FROM bookings b
                        WHERE b.status = 'confirmed')
ORDER BY g.guestid;
""", con=engine)
engine.dispose()
print(len(df_B3_orig), 'Zeilen')
df_B3_orig

**Aufgabe:** Erklären Sie, warum die Abfrage keine Zeilen liefert (Tipp: Welche Werte liefert die Subquery? Prüfen Sie sie separat). Schreiben Sie **zwei** korrekte Varianten: eine mit `NOT EXISTS` und eine als Anti-Join mit `LEFT JOIN … IS NULL`. Beide müssen gleich viele Zeilen liefern. Begründen Sie, wo bei der Anti-Join-Variante die Bedingung `status = 'confirmed'` stehen muss und warum. *(→ 4.3, 4.5, 11.1)*

**Begründung der Änderungen:**

> [Hier eintragen]

**Korrigierte SQL-Abfrage (Variante NOT EXISTS):**

In [ ]:
engine = create_engine(db_connection_url)

df_B3_a = pd.read_sql_query("""
-- Korrigierte SQL-Abfrage mit NOT EXISTS hier eintragen

""", con=engine)

engine.dispose()
df_B3_a

**Korrigierte SQL-Abfrage (Variante Anti-Join):**

In [ ]:
engine = create_engine(db_connection_url)

df_B3_b = pd.read_sql_query("""
-- Korrigierte SQL-Abfrage mit LEFT JOIN ... IS NULL hier eintragen

""", con=engine)

engine.dispose()
df_B3_b

---
## Teil C – SQL-Abfragen schreiben (20 Punkte)

*Schreiben Sie SQL-Abfragen und begründen Sie Ihre Vorgehensweise jeweils in 1–2 Sätzen.*

### C1 (4 Punkte)

Zeigen Sie zuerst das **Wertespektrum** von `member_type` (jeder vorkommende Wert mit Anzahl, inkl. `NULL`, absteigend nach Anzahl).

Schreiben Sie danach eine Abfrage, die alle Gäste (`guestid`, `firstname`, `lastname`, `member_type`) mit einem **ungültigen** Wert ausgibt – gültig sind nur exakt `Mitglied`, `Gast`, `Jugend`. Begründen Sie Ihre Filterbedingung (insbesondere den Umgang mit `NULL`). *(→ 4.3, 4.9)*

**Begründung der Vorgehensweise:**

> [Hier eintragen]

In [ ]:
engine = create_engine(db_connection_url)

df_C1_spectrum = pd.read_sql_query("""
-- SQL-Abfrage Wertespektrum hier eintragen

""", con=engine)

engine.dispose()
df_C1_spectrum

In [ ]:
engine = create_engine(db_connection_url)

df_C1 = pd.read_sql_query("""
-- SQL-Abfrage ungültige Werte hier eintragen

""", con=engine)

engine.dispose()
df_C1

### C2 (4 Punkte)

Berechnen Sie pro Hütte die **Stornoquote** in Prozent: Spalten `name`, `n_bookings` (alle Buchungen), `n_cancelled` und `cancel_rate_pct` (gerundet auf 1 Stelle mit `ROUND(CAST(... AS NUMERIC), 1)`). Berücksichtigen Sie nur Hütten mit **mindestens 200 Buchungen** und sortieren Sie absteigend nach `cancel_rate_pct`.

Begründen Sie, wie Sie die stornierten Buchungen zählen, warum `HAVING` statt `WHERE` nötig ist und wie Sie die Ganzzahl-Division vermeiden. *(→ 4.2, 4.4, 4.6)*

**Begründung der Vorgehensweise:**

> [Hier eintragen]

In [ ]:
engine = create_engine(db_connection_url)

df_C2 = pd.read_sql_query("""
-- SQL-Abfrage hier eintragen

""", con=engine)

engine.dispose()
df_C2

### C3 (4 Punkte)

Berechnen Sie die bestätigten Übernachtungen (`persons × nights`) **pro Monat und Hütte**. Verwenden Sie für den Monat `SUBSTR(arrival_date, 1, 7)` (funktioniert in PostgreSQL und SQLite) mit dem Alias `month`. Die SQL-Abfrage liefert das Ergebnis im **Long-Format** (`month`, `name`, `overnight_stays`); formen Sie es anschliessend mit pandas in eine Tabelle **Monat × Hütte** um (`pivot_table`, fehlende Werte = 0).

Begründen Sie, warum `SUBSTR` hier funktioniert und welche PostgreSQL-Funktion man verwenden würde, wenn `arrival_date` als `DATE` gespeichert wäre. *(→ 4.7, 6.9)*

**Begründung der Vorgehensweise:**

> [Hier eintragen]

In [ ]:
engine = create_engine(db_connection_url)

df_C3 = pd.read_sql_query("""
-- SQL-Abfrage hier eintragen

""", con=engine)

engine.dispose()
df_C3

In [ ]:
# pandas: Long -> Wide hier eintragen

### C4 (4 Punkte)

Ermitteln Sie pro Hütte die **Top-3-Gäste** nach Anzahl bestätigter Übernachtungen (`persons × nights`, summiert über alle Buchungen des Gastes in dieser Hütte). Bei Gleichstand sollen alle gleichrangigen Gäste erscheinen. Ausgabe: `hut`, `guestid`, `lastname`, `stays`, `rnk`; sortiert nach `hut` und `rnk`. Walk-in-Buchungen ohne `guestid` sollen nicht berücksichtigt werden.

Begründen Sie, warum Sie `RANK()` statt `ROW_NUMBER()` verwenden und warum die Window-Funktion in einer CTE berechnet und erst danach gefiltert wird. *(→ 4.6, 4.7)*

**Begründung der Vorgehensweise:**

> [Hier eintragen]

In [ ]:
engine = create_engine(db_connection_url)

df_C4 = pd.read_sql_query("""
-- SQL-Abfrage hier eintragen

""", con=engine)

engine.dispose()
df_C4

### C5 (4 Punkte)

Der Verband möchte Bewertungen erfassen. Erstellen Sie die Tabelle `reviews` mit den Spalten `reviewid` (Primärschlüssel), `bookingid` (Pflicht, Fremdschlüssel auf `bookings`), `stars` (Pflicht, nur 1–5 erlaubt), `comment` (Text, max. 500 Zeichen) und `created_at` (Text `JJJJ-MM-TT`, Pflicht).

Fügen Sie anschliessend **eine gültige** Bewertung ein und versuchen Sie, **eine ungültige** Bewertung mit `stars = 6` einzufügen. Fangen Sie den Fehler mit `try/except Exception as e` ab und geben Sie die Fehlermeldung aus. Zeigen Sie zum Schluss den Inhalt von `reviews`.

Begründen Sie, warum ein `CHECK`-Constraint besser ist als eine Prüfung im Python-Code der Erfassungsmaske. *(→ 4.1, 4.8)*

*Hinweis:* DDL/DML-Anweisungen führen Sie mit `with engine.begin() as connection: connection.execute(text("..."))` aus – `begin()` schreibt die Änderung am Ende des Blocks fest (Commit).

**Begründung der Vorgehensweise:**

> [Hier eintragen]

In [ ]:
# Code hier eintragen

# Create table

# Insert a valid review

# Try to insert an invalid review (stars = 6)

# Show the table

---
### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
# Libraries
import os, platform, socket
from platform import python_version
from datetime import datetime

# Print system information
print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
print('Python Version:', python_version())
print('-----------------------------------')